# rx3 GLiNER LoRA retraining (Kaggle)
Port of `docs/RETRAINING.md` steps 2-4. Needs: GPU (P100 or T4), Internet ON (downloads `fastino/gliner2.5-base-v1`), and the dataset built by `kaggle/make_kaggle_inputs.py`.
Outputs in `/kaggle/working`: `rx3-entry-lora/` (the new adapter), `rx3-entry-lora.zip`, `eval_*.txt`.

In [ ]:
# ---- settings ---------------------------------------------------------------
REBUILD_DATA = True    # True: rebuild data/train from PDFs (RETRAINING step 2). False: use the prebuilt data/train/*.jsonl as uploaded
MAX_SYNTH    = 8000    # --max-synth
USE_LLM_JSON = True    # --llm-json: also label the ~1,450 LiveCareer resumes outside the HTML-tag pools from resume-data/JSONs (verbatim values only)
MAX_LC       = 6000    # --max-lc  cap on LiveCareer blocks (HTML-tag labels first, LLM-JSON fills the rest). Lower it if gold/test regress (LC drift)
EPOCHS       = 3
BATCH        = 8       # P100/T4 16 GB. If CUDA OOM: BATCH=4, ACCUM=4
ACCUM        = 2
LR           = 1e-4    # encoder LoRA lr
TASK_LR      = 2e-4    # task-head lr (5e-4 gave non-finite losses in fp16)
RANK         = 8
SEED         = 0
RUN_EVAL     = True    # compare new adapter vs the shipped one on held-out splits

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
from pathlib import Path

def sh(cmd, **kw):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    for line in p.stdout: print(line, end="", flush=True)
    rc = p.wait()
    if rc: raise RuntimeError(f"exit {rc}: {cmd}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv || echo NO GPU - enable a GPU accelerator in the notebook settings")

# locate the uploaded dataset (Kaggle auto-extracts the zip; handle both)
WORK = Path("/kaggle/working/IPD")
if not WORK.exists():
    hits = glob.glob("/kaggle/input/*/IPD") + glob.glob("/kaggle/input/*/*/IPD")
    if hits:
        shutil.copytree(hits[0], WORK)
    else:
        z = (glob.glob("/kaggle/input/**/rx3-kaggle-inputs.zip", recursive=True) or [None])[0]
        assert z, "dataset not found under /kaggle/input - add it via 'Add Input'"
        zipfile.ZipFile(z).extractall("/kaggle/working")
RX3 = WORK / "resume-extractor-v3"
assert (RX3 / "tools" / "train_gliner_lora.py").exists(), "bad dataset layout"
print("project:", RX3)

In [ ]:
# deps pinned to uv.lock (gliner2 2.0.0 / peft 0.21.2 / transformers 4.57.6); torch is Kaggle's preinstalled CUDA build
sh("pip install -q 'gliner2[train]==2.0.0' 'peft==0.21.2' 'transformers==4.57.6' datasets scikit-learn "
   "pymupdf python-docx pydantic dateparser rapidfuzz phonenumbers ftfy flashtext numpy pytesseract")
sh("pip install -q rapidocr-onnxruntime || echo 'rapidocr unavailable - only needed for scanned PDFs'")
import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU"

In [ ]:
os.environ["PYTHONPATH"] = f"{RX3/'src'}:{RX3/'tools'}"
os.chdir(RX3)
if REBUILD_DATA:
    # LiveCareer labels come from data/livecareer/aligned_train.jsonl (already aligned to the HTML tags; disjoint from the eval pool aligned.jsonl)
    sh(f"python tools/build_gliner_data.py --max-synth {MAX_SYNTH} --max-lc {MAX_LC} --seed {SEED}" + (" --llm-json" if USE_LLM_JSON else ""))
for f in ("train", "dev", "dev_lc", "test_synth"):
    p = RX3 / "data" / "train" / f"{f}.jsonl"
    print(f, sum(1 for _ in open(p, encoding="utf-8")) if p.exists() else "MISSING")

In [ ]:
OUT = "/kaggle/working/gliner-lora-new"
sh(f"python -W ignore tools/train_gliner_lora.py --epochs {EPOCHS} --batch {BATCH} --accum {ACCUM} --lr {LR} --task-lr {TASK_LR} --rank {RANK} --out {OUT}")
print(sorted(os.listdir(OUT)))

In [ ]:
# eval on held-out data: new adapter vs the shipped one (zero-shot baseline is printed too)
if RUN_EVAL:
    NEW, OLD = f"{OUT}/best", str(RX3 / "models/adapters/rx3-entry-lora")
    for split in ("test_synth", "dev_lc"):
        for tag, ad in (("new", NEW), ("shipped", OLD)):
            if Path(ad).exists():
                sh(f"python -W ignore -m eval.blocks_eval --adapter {ad} --split {split} --device cuda 2>&1 | tee /kaggle/working/eval_{split}_{tag}.txt")

In [ ]:
# package the new adapter; compare with the shipped one before using it (RETRAINING step 4: no regression on test_synth / dev_lc / gold32)
import hashlib
dst = Path("/kaggle/working/rx3-entry-lora")
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(f"{OUT}/best", dst)
for f in dst.iterdir(): print(f.name, f.stat().st_size)
print("sha256 adapter_model.safetensors:", hashlib.sha256((dst / "adapter_model.safetensors").read_bytes()).hexdigest())
shutil.make_archive("/kaggle/working/rx3-entry-lora", "zip", dst)
shutil.rmtree(WORK, ignore_errors=True)   # keep /kaggle/working small (output = adapter + eval logs only)
shutil.rmtree(OUT, ignore_errors=True)